In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install timm

In [ ]:
import os
import cv2
import torch
import timm
import numpy as np
from tqdm import tqdm
import torchvision.transforms as transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
print("EfficientNet-B4 Available:",
      "efficientnet_b4" in timm.list_models())

print("Xception Available:",
      "xception" in timm.list_models())

In [ ]:
!pip -q install pretrainedmodels

In [ ]:
import pretrainedmodels
import torch.nn as nn

In [ ]:
efficientnet = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=0
)

efficientnet = efficientnet.to(device)
efficientnet.eval()

print("EfficientNet-B4 Loaded Successfully")

In [ ]:
import tensorflow as tf

from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.xception import preprocess_input

xception = Xception(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

print("Xception Loaded")

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

INPUT_PATH = os.path.join(PROJECT_PATH, "dwt_fusion_images")

OUTPUT_PATH = os.path.join(PROJECT_PATH, "spatial_features")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Deepfakes"), exist_ok=True)

In [ ]:
sample_video = sorted(os.listdir(os.path.join(INPUT_PATH, "original")))[0]

sample_frame = sorted(
    os.listdir(
        os.path.join(INPUT_PATH, "original", sample_video)
    )
)[0]

image_path = os.path.join(
    INPUT_PATH,
    "original",
    sample_video,
    sample_frame
)

print(image_path)

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
import torch.nn as nn

projection = nn.Linear(3840,512).to(device)

projection.eval()

In [ ]:
def extract_spatial_features(image_path):

    # ---------- EfficientNet ----------
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        eff = efficientnet(img_tensor)

    eff = eff.cpu().numpy().flatten()


    # ---------- Xception ----------
    img = cv2.resize(image,(299,299))

    img = np.expand_dims(img,0)

    img = preprocess_input(img.astype(np.float32))

    x = xception.predict(img,verbose=0).flatten()


    # ---------- Concatenate ----------
    feature = np.concatenate([eff,x])

    feature = torch.tensor(
        feature,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = projection(feature)

    embedding = embedding.squeeze(0).cpu().numpy()

    return embedding

In [ ]:
feature = extract_spatial_features(image_path)

print(feature.shape)

In [ ]:
SPATIAL_REAL = os.path.join(PROJECT_PATH, "spatial_features", "original")
SPATIAL_FAKE = os.path.join(PROJECT_PATH, "spatial_features", "Deepfakes")

os.makedirs(SPATIAL_REAL, exist_ok=True)
os.makedirs(SPATIAL_FAKE, exist_ok=True)

In [ ]:
def process_video(video_folder):

    frames = sorted(os.listdir(video_folder))

    embeddings = []

    for frame in frames:

        image_path = os.path.join(video_folder, frame)

        emb = extract_spatial_features(image_path)

        embeddings.append(emb)

    embeddings = np.array(embeddings)

    return embeddings

In [ ]:
sample_video = sorted(
    os.listdir(os.path.join(INPUT_PATH, "original"))
)[0]

video_path = os.path.join(
    INPUT_PATH,
    "original",
    sample_video
)

video_feature = process_video(video_path)

print("Video Shape :", video_feature.shape)

In [ ]:
print(video_feature)

In [ ]:
def process_spatial_dataset(input_root, output_root):

    processed = 0

    videos = sorted(os.listdir(input_root))

    for video in tqdm(videos):

        video_path = os.path.join(input_root, video)

        feature = process_video(video_path)

        np.save(
            os.path.join(output_root, video + ".npy"),
            feature
        )

        processed += 1

    print("=" * 50)
    print("Completed")
    print("Processed Videos :", processed)

In [ ]:
process_spatial_dataset(
    os.path.join(INPUT_PATH, "original"),
    SPATIAL_REAL
)

In [ ]:
process_spatial_dataset(
    os.path.join(INPUT_PATH, "Deepfakes"),
    SPATIAL_FAKE
)

In [ ]:
sample = sorted(os.listdir(SPATIAL_REAL))[0]

feature = np.load(
    os.path.join(SPATIAL_REAL, sample)
)

print(feature.shape)